# Split CSV 분석

`SPLIT_CSV` 경로를 지정하면 가장 마지막 R 컬럼의 파티션별 선택 분포를 시각화합니다.

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.cm as cm
import numpy as np

SPLIT_CSV = "../checkpoints/stage2-v4-fedpid-entropy-0/agg/init/split.csv"

df = pd.read_csv(SPLIT_CSV)

# 정수 컬럼 Int64 복원
df["Partition_ID"] = df["Partition_ID"].astype("Int64")
for col in df.columns:
    if col.startswith("R") and col[1:].isdigit():
        df[col] = df[col].astype("Int64")
if "pool" in df.columns:
    df["pool"] = df["pool"].astype("Int64")

# 마지막 R 컬럼 자동 탐색
r_cols = sorted([c for c in df.columns if c.startswith("R") and c[1:].isdigit()])
last_r = r_cols[-1] if r_cols else None

# pool 인코딩 감지
has_pool   = "pool" in df.columns
train_df   = df[df["TrainOrVal"] == "train"]
# entropy pool: 전체 train subject에 순위(1..N) 부여 → max > 1 & pool=0 없음
is_ranked  = has_pool and int(train_df["pool"].max(skipna=True)) > 1 and int((train_df["pool"] == 0).sum()) == 0

print(f"CSV        : {SPLIT_CSV}")
print(f"Rows       : {len(df)}  (train={len(train_df)}, val={len(df)-len(train_df)})")
print(f"R columns  : {r_cols}  →  plotting: {last_r}")
print(f"Pool type  : {'ranked 1..N (entropy/anti_entropy)' if is_ranked else 'binary 0/1 (random)' if has_pool else 'none'}")
df.head()

In [ ]:
if is_ranked:
    # ── ranked pool (entropy/anti_entropy) ───────────────────────────────────
    # pool 컬럼 = 전체 N개 BALD 순위 (1=최고 BALD, N=최저 BALD)
    # "pool vs non-pool" 대신 R00 선택/비선택으로 구분
    N = int(train_df["pool"].max(skipna=True))
    partitions = sorted(train_df["Partition_ID"].dropna().unique())
    x_labels = [f"P{p}" for p in partitions]

    has_r00 = "R00" in train_df.columns
    sel_df  = train_df[train_df["R00"] == 1] if has_r00 else None
    non_df  = train_df[train_df["R00"] == 0] if has_r00 else None

    if has_r00:
        fig, axes = plt.subplots(1, 2, figsize=(16, 4))

        # 왼쪽: 파티션별 R00 선택/비선택 stacked bar
        sel_counts = sel_df.groupby("Partition_ID").size().reindex(partitions, fill_value=0)
        non_counts = non_df.groupby("Partition_ID").size().reindex(partitions, fill_value=0)
        total_sel  = int(sel_counts.sum())
        ax = axes[0]
        b1 = ax.bar(x_labels, sel_counts.values, color="steelblue", edgecolor="white", label=f"R00 selected ({total_sel})")
        b2 = ax.bar(x_labels, non_counts.values, bottom=sel_counts.values,
                    color="lightgray", edgecolor="white", label=f"not selected ({N - total_sel})")
        ax.bar_label(b1, labels=[str(v) if v > 0 else "" for v in sel_counts.values],
                     label_type="center", fontsize=8, color="white")
        ax.set_title(f"R00 selected vs not  (total train={N})", fontsize=11)
        ax.set_xlabel("Partition"); ax.set_ylabel("Subjects")
        ax.set_ylim(0, (sel_counts + non_counts).max() * 1.2)
        ax.legend(loc="upper right"); ax.grid(axis="y", linestyle="--", alpha=0.4)
        ax.tick_params(axis="x", labelsize=7)

        # 오른쪽: 파티션별 선택된 subjects의 BALD 순위 중위수
        # 낮은 순위(초록) = 고BALD 선택, 높은 순위(빨강) = 저BALD 선택
        med_sel = sel_df.groupby("Partition_ID")["pool"].median().reindex(partitions)
        med_non = non_df.groupby("Partition_ID")["pool"].median().reindex(partitions)
        colors_sel = cm.RdYlGn_r(med_sel.values / N)
        ax = axes[1]
        x = np.arange(len(partitions))
        w = 0.4
        bars_sel = ax.bar(x - w/2, med_sel.values, width=w, color=colors_sel, edgecolor="white", label="selected median rank")
        bars_non = ax.bar(x + w/2, med_non.values, width=w, color="lightgray", edgecolor="white", label="not selected median rank")
        ax.bar_label(bars_sel, labels=[f"{v:.0f}" if not np.isnan(v) else "" for v in med_sel.values],
                     padding=2, fontsize=7)
        ax.axhline(N / 2, color="gray", linestyle="--", linewidth=0.8, label=f"global median={N//2}")
        ax.set_title("Median BALD rank: selected vs not  (1=highest BALD)", fontsize=11)
        ax.set_xlabel("Partition"); ax.set_ylabel("Median rank")
        ax.set_xticks(x); ax.set_xticklabels(x_labels, fontsize=7)
        ax.set_ylim(0, N * 1.1)
        ax.legend(fontsize=8); ax.grid(axis="y", linestyle="--", alpha=0.4)
    else:
        # R00 없음: BALD 순위 분포만 표시
        fig, ax = plt.subplots(figsize=(10, 4))
        medians = train_df.groupby("Partition_ID")["pool"].median()
        colors  = cm.RdYlGn_r(medians.values / N)
        bars = ax.bar(x_labels, medians.values, color=colors, edgecolor="white")
        ax.bar_label(bars, labels=[f"{v:.0f}" for v in medians.values], padding=3, fontsize=8)
        ax.axhline(N / 2, color="gray", linestyle="--", linewidth=0.8, label=f"global median={N//2}")
        ax.set_title(f"Median BALD rank per Partition  (N={N}, 1=highest BALD)", fontsize=11)
        ax.set_xlabel("Partition"); ax.set_ylabel("Median rank")
        ax.set_ylim(0, N * 1.1)
        ax.legend(fontsize=9); ax.grid(axis="y", linestyle="--", alpha=0.4)
        ax.tick_params(axis="x", labelsize=7)

    plt.suptitle(f"{SPLIT_CSV}", fontsize=9)
    plt.tight_layout()
    plt.show()

    # 텍스트 테이블
    print(f"\n{'Partition':>12}  {'total':>6}  {'R00_sel':>8}  {'rank_min':>9}  {'rank_med_sel':>13}  {'rank_med_non':>13}  {'rank_max':>9}")
    for p in partitions:
        pf  = train_df[train_df["Partition_ID"] == p]
        sf  = pf[pf["R00"] == 1] if has_r00 else pf
        nf  = pf[pf["R00"] == 0] if has_r00 else None
        med_s = f"{sf['pool'].median():.1f}" if len(sf) > 0 else "-"
        med_n = f"{nf['pool'].median():.1f}" if (nf is not None and len(nf) > 0) else "-"
        print(f"{p:>12}  {len(pf):>6}  {len(sf):>8}  {int(pf['pool'].min()):>9}  {med_s:>13}  {med_n:>13}  {int(pf['pool'].max()):>9}")

elif has_pool:
    # ── binary pool (random): pool>0 / pool=0 stacked bar ───────────────────
    all_counts  = train_df.groupby("Partition_ID").size()
    pool_counts = train_df[train_df["pool"] > 0].groupby("Partition_ID").size().reindex(all_counts.index, fill_value=0)
    nonp_counts = all_counts - pool_counts
    x_labels = [f"P{p}" for p in all_counts.index]

    fig, ax = plt.subplots(figsize=(10, 3))
    b1 = ax.bar(x_labels, pool_counts.values, color="steelblue", edgecolor="white", label=f"pool ({int(pool_counts.sum())})")
    b2 = ax.bar(x_labels, nonp_counts.values, bottom=pool_counts.values,
                color="lightcoral", edgecolor="white", label=f"non-pool ({int(nonp_counts.sum())})")
    ax.bar_label(b1, labels=[str(v) if v > 0 else "" for v in pool_counts.values],
                 label_type="center", fontsize=8, color="white")
    ax.set_title(f"Pool membership per Partition  (total train={len(train_df)})", fontsize=11)
    ax.set_xlabel("Partition"); ax.set_ylabel("Subjects")
    ax.set_ylim(0, all_counts.max() * 1.2)
    ax.legend(loc="upper right"); ax.grid(axis="y", linestyle="--", alpha=0.4)
    ax.tick_params(axis="x", labelsize=7)
    plt.tight_layout(); plt.show()

else:
    print("pool 컬럼 없음 — pool 시각화 생략")

In [ ]:
if not r_cols:
    print("R 컬럼 없음 — 선택 분포 시각화 생략")
else:
    # pool 필터: ranked → 전체 train, binary → pool>0, 없음 → 전체 train
    if is_ranked:
        plot_df = train_df          # 전체 train (모두 순위 보유)
        pool_label = "all train (ranked pool)"
    elif has_pool:
        plot_df = train_df[train_df["pool"] > 0]
        pool_label = "pool subjects"
    else:
        plot_df = train_df
        pool_label = "all train"

    fig, axes = plt.subplots(len(r_cols), 1, figsize=(10, 3 * len(r_cols)))
    if len(r_cols) == 1:
        axes = [axes]

    for ax, r_col in zip(axes, r_cols):
        selected     = plot_df.groupby("Partition_ID")[r_col].sum()
        total_sel    = int(selected.sum())
        total_avail  = len(plot_df)
        bars = ax.bar([f"P{p}" for p in selected.index], selected.values,
                      color="steelblue", edgecolor="white")
        ax.bar_label(bars, padding=4, fontsize=9)
        ax.set_title(f"{r_col}  (selected={total_sel} / {pool_label}={total_avail})", fontsize=11)
        ax.set_xlabel("Partition"); ax.set_ylabel("Selected")
        ax.set_ylim(0, selected.max() * 1.25 if selected.max() > 0 else 1)
        ax.grid(axis="y", linestyle="--", alpha=0.4)
        ax.tick_params(axis="x", labelsize=7)

    fig.suptitle(f"{SPLIT_CSV}", fontsize=9)
    plt.tight_layout()
    plt.show()

    # 텍스트 테이블
    print(f"\n{'Partition':>12}", end="")
    for r_col in r_cols:
        print(f"  {r_col:>10}", end="")
    print()
    for p in sorted(plot_df["Partition_ID"].dropna().unique()):
        pf = plot_df[plot_df["Partition_ID"] == p]
        print(f"{p:>12}", end="")
        for r_col in r_cols:
            n_avail = len(pf)
            n_sel   = int(pf[r_col].sum())
            print(f"  {n_sel:>4}/{n_avail:<4}", end="")
        print()